# Notebook 04: Full End-to-End Fine-Tuning

In this notebook, we perform true end-to-end training. **All weights of the Moirai encoder and the classification heads are updated**.


In [1]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import copy
from IPython.display import display

from tslearn.datasets import UCR_UEA_datasets
from sklearn.preprocessing import LabelEncoder
from uni2ts.model.moirai import MoiraiModule
from encoder import MoiraiEncoder

from heads import *
from models import *
from utils import  *

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
NUM_VARS = 6
SIZE = "small"
PATCH_SIZES = [8, 16, 32, 64]

lr_grid=[1e-4]
wd_grid=[0.05]

BATCH_SIZE = 256

/home/onyxia/work/DLforTimeSeries/moirai-classification/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tr_loader, va_loader, te_loader, num_classes = get_lsst_dataloaders(BATCH_SIZE, DEVICE)

In [3]:
METRICS_COLS = ["Accuracy", "Macro Precision", "Macro Recall", "Macro F1", "Weighted Precision", "Weighted Recall", "Weighted F1"]
df_patch_8_metrics = pd.DataFrame(columns=METRICS_COLS)
df_patch_8_metrics.index.name = "Model"

## 1. Baseline: Linear Model on Mask Embedding Only (Full FT)

In [4]:
df_mask_only = pd.DataFrame(index=["Mask Only"], columns=PATCH_SIZES)
df_mask_only.columns.name = "Patch Size"

for patch in PATCH_SIZES:
    _, metrics = universal_grid_search(
        model_class=FullMaskOnlyWrapper,
        model_kwargs={"patch_size": patch, "num_vars": NUM_VARS, "num_classes": num_classes, "size": SIZE},
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
        wd_grid=wd_grid, lr_grid=lr_grid, verbose=True
    )
    df_mask_only.loc["Mask Only", patch] = metrics["Accuracy"]
    if patch == 8:
        df_patch_8_metrics.loc["Mask Only"] = metrics

LR=0.0001| WD=0.05
1.8154859097023321
1.4946233799787072
1.3497064646666612
1.2532337419385833
1.212744335818097
1.1860913716680634
1.1744311330764274
1.2019058446574018
1.214283249242519
1.285621485089868
1.289043067916622
1.3211739673847105
 Early stopping : 11
Val Loss: 1.1744
LR=0.0001| WD=0.05
1.806666280195965
1.5223553674977
1.3537399216396053
1.2792758108154545
1.2521767102605927
1.2321503879578133
1.2576983905420072
1.314900387593401
1.326508004490922
1.394312181123873
1.427372438151662
 Early stopping : 10
Val Loss: 1.2322
LR=0.0001| WD=0.05
1.7792946575133781
1.491369708766782
1.3528182021970672
1.287336022873235
1.228137376831799
1.243063633034869
1.2630347972962914
1.2899045430547822
1.3466770387277371
1.4197385572805636
 Early stopping : 9
Val Loss: 1.2281
LR=0.0001| WD=0.05
1.6841828881240473
1.4120557628026822
1.2741529699263534
1.2645053659997336
1.2225948077876394
1.1934324493253134
1.2150062826590808
1.232910577843829
1.272711628820838
1.3422050359772473
1.3415795428

In [5]:
print("Mask Only - Accuracy")
display(df_mask_only.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))

Mask Only - Accuracy


Patch Size,8,16,32,64
Mask Only,0.6196,0.6188,0.6192,0.6099



Patch = 8 — All metrics


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Model,,,,,,,
Mask Only,0.6196,0.4389,0.3398,0.3444,0.5618,0.6196,0.5624


## 2. Baseline: Mean Pooling

In [6]:
df_mean_pool = pd.DataFrame(index=["Mean Pooling"], columns=PATCH_SIZES)
df_mean_pool.columns.name = "Patch Size"

for patch in PATCH_SIZES:
    _, metrics = universal_grid_search(
        model_class=FullHeadWrapper,
        model_kwargs={
            "head_class": MeanPoolingClassifier,
            "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes},
            "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
        },
        train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
        wd_grid=wd_grid, lr_grid=lr_grid, verbose=True
    )
    df_mean_pool.loc["Mean Pooling", patch] = metrics["Accuracy"]
    if patch == 8:
        df_patch_8_metrics.loc["Mean Pooling"] = metrics

LR=0.0001| WD=0.05
1.7443417814688953
1.464175550918269
1.307793854697933
1.2296836773554485
1.1792836305571766
1.1108497885184558
1.1063486221359997
1.092919442711807
1.089577285254874
1.0918856587836412
1.1226664471432446
1.168960978345173
1.1996408574949435
1.2654931496798507
 Early stopping : 13
Val Loss: 1.0896
LR=0.0001| WD=0.05
1.7948515919165882
1.5026338565640334
1.3294052486497212
1.2087256113688152
1.1526291331624596
1.1192503586048033
1.0906712010623962
1.088459830943162
1.1372523230265796
1.1328330243506082
1.1152661592979742
1.1912941254251372
1.1941118094979264
 Early stopping : 12
Val Loss: 1.0885
LR=0.0001| WD=0.05
1.7367689783980207
1.4641395894492544
1.3026943042026302
1.2389535651943548
1.1509809552169428
1.1215324140176541
1.1110431944451682
1.1251201639330484
1.1493246981768104
1.208758041141479
1.2381882522164323
1.271044704972244
 Early stopping : 11
Val Loss: 1.1110
LR=0.0001| WD=0.05
1.615010870181448
1.3810684351417106
1.2481141090393066
1.1911004490968657
1.

In [7]:
print("Mean Pooling - Accuracy")
display(df_mean_pool.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))

Mean Pooling - Accuracy


Patch Size,8,16,32,64
Mean Pooling,0.6565,0.6383,0.6338,0.6285



Patch = 8 — All metrics


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Model,,,,,,,
Mask Only,0.6196,0.4389,0.3398,0.3444,0.5618,0.6196,0.5624
Mean Pooling,0.6565,0.4276,0.3789,0.3802,0.5860,0.6565,0.6031


## 3. Advanced Pooling: Attention & MHA (Full FT)

In [8]:
PATCH_SIZES_TO_TEST = [8, 16, 32, 64]
MODES = ["shared_context", "independent_context"]
NUM_HEADS = 16

model_names_single = ["Attention (Base)", f"MHA (Heads={NUM_HEADS})"]
index_single = pd.MultiIndex.from_product([model_names_single, MODES], names=["Model", "Mode"])
df_adv_single = pd.DataFrame(index=index_single, columns=PATCH_SIZES_TO_TEST)
df_adv_single.columns.name = "Patch Size"

for patch in PATCH_SIZES_TO_TEST:
    for mode in MODES:
        # 1. Basic Attention
        _, metrics_attn = universal_grid_search(
            model_class=FullHeadWrapper,
            model_kwargs={
                "head_class": SingleScaleAttentionClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "mode": mode},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            wd_grid=wd_grid, lr_grid=lr_grid
        )
        df_adv_single.loc[("Attention (Base)", mode), patch] = metrics_attn["Accuracy"]
        if patch == 8:
            df_patch_8_metrics.loc[f"Attention ({mode})"] = metrics_attn

        # 2. Multi-Head Attention
        _, metrics_mha = universal_grid_search(
            model_class=FullHeadWrapper,
            model_kwargs={
                "head_class": SingleScaleMultiHeadClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": NUM_HEADS},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            wd_grid=wd_grid, lr_grid=lr_grid
        )
        df_adv_single.loc[(f"MHA (Heads={NUM_HEADS})", mode), patch] = metrics_mha["Accuracy"]

print("Attention & MHA - Accuracy")
display(df_adv_single.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))

LR=0.0001| WD=0.05
 Early stopping : 12
Val Loss: 1.1594
LR=0.0001| WD=0.05
 Early stopping : 12
Val Loss: 1.0785
LR=0.0001| WD=0.05
 Early stopping : 12
Val Loss: 1.1399
LR=0.0001| WD=0.05
 Early stopping : 13
Val Loss: 1.0694
LR=0.0001| WD=0.05
 Early stopping : 11
Val Loss: 1.1816
LR=0.0001| WD=0.05
 Early stopping : 10
Val Loss: 1.1092
LR=0.0001| WD=0.05
 Early stopping : 10
Val Loss: 1.1510
LR=0.0001| WD=0.05
 Early stopping : 11
Val Loss: 1.1321
LR=0.0001| WD=0.05
 Early stopping : 13
Val Loss: 1.1361
LR=0.0001| WD=0.05
 Early stopping : 12
Val Loss: 1.1215
LR=0.0001| WD=0.05
 Early stopping : 10
Val Loss: 1.1762
LR=0.0001| WD=0.05
 Early stopping : 9
Val Loss: 1.1665
LR=0.0001| WD=0.05
 Early stopping : 12
Val Loss: 1.1815
LR=0.0001| WD=0.05
 Early stopping : 11
Val Loss: 1.1430
LR=0.0001| WD=0.05
 Early stopping : 9
Val Loss: 1.1711
LR=0.0001| WD=0.05
 Early stopping : 10
Val Loss: 1.1519
Attention & MHA - Accuracy


Patch Size                                8       16      32      64
Model            Mode                                               
Attention (Base) shared_context       0.6269  0.6375  0.6237  0.6221
                 independent_context  0.6387  0.6395  0.6107  0.6403
MHA (Heads=16)   shared_context       0.6448  0.6164  0.6233  0.6261
                 independent_context  0.6460  0.6123  0.6176  0.6245


Patch = 8 — All metrics


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Model,,,,,,,
Mask Only,0.6196,0.4389,0.3398,0.3444,0.5618,0.6196,0.5624
Mean Pooling,0.6565,0.4276,0.3789,0.3802,0.5860,0.6565,0.6031
Attention (shared_context),0.6269,0.4200,0.3758,0.3693,0.5634,0.6269,0.5781
Attention (independent_context),0.6387,0.4777,0.3733,0.3738,0.5791,0.6387,0.5848


## 4. Ablation Study: Number of Attention Heads

In [9]:
HEADS_TO_TEST = [16, 32, 64, 128]

df_heads_ablation8 = pd.DataFrame(index=MODES, columns=HEADS_TO_TEST)
df_heads_ablation8.index.name = "Mode"
df_heads_ablation8.columns.name = "Num Heads (Patch 8)"
df_heads_ablation16 = pd.DataFrame(index=MODES, columns=HEADS_TO_TEST)
df_heads_ablation16.index.name = "Mode"
df_heads_ablation16.columns.name = "Num Heads (Patch 16)"

for PATCH in [8, 16]:
    for mode in MODES:
        for heads in HEADS_TO_TEST:
            _, metrics = universal_grid_search(
                model_class=FullHeadWrapper,
                model_kwargs={
                    "head_class": SingleScaleMultiHeadClassifier,
                    "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": heads},
                    "patch_size": PATCH, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
                },
                train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
                wd_grid=wd_grid, lr_grid=lr_grid, verbose=True
            )
            if PATCH == 8:
                df_heads_ablation8.loc[mode, heads] = metrics["Accuracy"]
                df_patch_8_metrics.loc[f"MHA-{heads} ({mode})"] = metrics
            else:
                df_heads_ablation16.loc[mode, heads] = metrics["Accuracy"]

LR=0.0001| WD=0.05
1.8792030946995184
1.587150946865237
1.3727774503754406
1.2533731005056117
1.2024532643760122
1.1295714349281498
1.1120942540285064
1.0980616110127146
1.1063881240239957
1.1157859486293018
1.1964236391269094
1.2842744298097564
1.4340674722097753
 Early stopping : 12
Val Loss: 1.0981
LR=0.0001| WD=0.05
1.9495068003491658
1.574629487060919
1.3791399632042987
1.3061113260625823
1.2286659430682174
1.1426450266101496
1.1761039224097398
1.1324723590680255
1.096756697670231
1.1422289600217246
1.1594579966087653
1.2534034320009433
1.2797474570390655
1.4661017792011664
 Early stopping : 13
Val Loss: 1.0968
LR=0.0001| WD=0.05
1.9351809577244083
1.554339984568154
1.35819765998096
1.2449165747417668
1.1559496265116747
1.1580696832842943
1.1797341573529128
1.1181464563540326
1.1252768272306861
1.187676118641365
1.1802993809304587
1.2340882464153011
1.3137529661984948
 Early stopping : 12
Val Loss: 1.1181
LR=0.0001| WD=0.05
1.9458519685559157
1.5944017637066725
1.3478356309053374


In [10]:
print("Ablation: Num Heads (Patch = 8) - Accuracy")
display(df_heads_ablation8.astype(float).round(4))
print("Ablation: Num Heads (Patch = 16) - Accuracy")
display(df_heads_ablation16.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))

Ablation: Num Heads (Patch = 8) - Accuracy


Num Heads (Patch 8),16,32,64,128
Mode,,,,
shared_context,0.6237,0.6480,0.6277,0.6334
independent_context,0.6371,0.6436,0.6403,0.6391


Ablation: Num Heads (Patch = 16) - Accuracy


Num Heads (Patch 16),16,32,64,128
Mode,,,,
shared_context,0.6273,0.6310,0.6419,0.6290
independent_context,0.6277,0.6395,0.6375,0.6391



Patch = 8 — All metrics


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Model,,,,,,,
Mask Only,0.6196,0.4389,0.3398,0.3444,0.5618,0.6196,0.5624
Mean Pooling,0.6565,0.4276,0.3789,0.3802,0.5860,0.6565,0.6031
Attention (shared_context),0.6269,0.4200,0.3758,0.3693,0.5634,0.6269,0.5781
Attention (independent_context),0.6387,0.4777,0.3733,0.3738,0.5791,0.6387,0.5848
MHA-16 (shared_context),0.6237,0.3861,0.3634,0.3552,0.5479,0.6237,0.5652
MHA-32 (shared_context),0.6480,0.4111,0.3815,0.3790,0.5760,0.6480,0.5950
MHA-64 (shared_context),0.6277,0.4512,0.3739,0.3780,0.5675,0.6277,0.5691
MHA-128 (shared_context),0.6334,0.4648,0.3835,0.3834,0.5869,0.6334,0.5977
MHA-16 (independent_context),0.6371,0.4089,0.3724,0.3731,0.5707,0.6371,0.5851


## 5. Hierarchical MHA (Full FT)

In [11]:
df_hierarchical = pd.DataFrame(index=MODES, columns=PATCH_SIZES_TO_TEST)
df_hierarchical.index.name = "Mode"
df_hierarchical.columns.name = "Patch Size"

for patch in PATCH_SIZES_TO_TEST:
    for mode in MODES:
        _, metrics_hier = universal_grid_search(
            model_class=FullHeadWrapper,
            model_kwargs={
                "head_class": HierarchicalMultiHeadClassifier,
                "head_kwargs": {"num_vars": NUM_VARS, "num_classes": num_classes, "patch_mode": mode, "num_heads": NUM_HEADS},
                "patch_size": patch, "num_vars": NUM_VARS, "size": SIZE, "remove_last_patch": False
            },
            train_loader=tr_loader, val_loader=va_loader, test_loader=te_loader, device=DEVICE,
            wd_grid=wd_grid, lr_grid=lr_grid
        )
        df_hierarchical.loc[mode, patch] = metrics_hier["Accuracy"]
        if patch == 8:
            df_patch_8_metrics.loc[f"Hierarchical ({mode})"] = metrics_hier

LR=0.0001| WD=0.05
 Early stopping : 15
Val Loss: 1.2529
LR=0.0001| WD=0.05
 Early stopping : 13
Val Loss: 1.2223
LR=0.0001| WD=0.05
 Early stopping : 13
Val Loss: 1.2558
LR=0.0001| WD=0.05
 Early stopping : 12
Val Loss: 1.2814
LR=0.0001| WD=0.05
 Early stopping : 12
Val Loss: 1.2924
LR=0.0001| WD=0.05
 Early stopping : 14
Val Loss: 1.2507
LR=0.0001| WD=0.05
 Early stopping : 13
Val Loss: 1.2847
LR=0.0001| WD=0.05
 Early stopping : 11
Val Loss: 1.2634


In [12]:
print("Hierarchical MHA - Accuracy")
display(df_hierarchical.astype(float).round(4))
print("\nPatch = 8 — All metrics")
display(df_patch_8_metrics.astype(float).round(4))

Hierarchical MHA - Accuracy


Patch Size,8,16,32,64
Mode,,,,
shared_context,0.6148,0.5994,0.5795,0.5876
independent_context,0.5973,0.5848,0.5904,0.5961



Patch = 8 — All metrics


,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted Precision,Weighted Recall,Weighted F1
Model,,,,,,,
Mask Only,0.6196,0.4389,0.3398,0.3444,0.5618,0.6196,0.5624
Mean Pooling,0.6565,0.4276,0.3789,0.3802,0.5860,0.6565,0.6031
Attention (shared_context),0.6269,0.4200,0.3758,0.3693,0.5634,0.6269,0.5781
Attention (independent_context),0.6387,0.4777,0.3733,0.3738,0.5791,0.6387,0.5848
MHA-16 (shared_context),0.6237,0.3861,0.3634,0.3552,0.5479,0.6237,0.5652
MHA-32 (shared_context),0.6480,0.4111,0.3815,0.3790,0.5760,0.6480,0.5950
MHA-64 (shared_context),0.6277,0.4512,0.3739,0.3780,0.5675,0.6277,0.5691
MHA-128 (shared_context),0.6334,0.4648,0.3835,0.3834,0.5869,0.6334,0.5977
MHA-16 (independent_context),0.6371,0.4089,0.3724,0.3731,0.5707,0.6371,0.5851
